In [ ]:
# Import all necessary modules
import os
import time
import glob
import shutil
import cv2
import PIL
import numpy as np
import pandas as pd
import seaborn as sns
sns.set_style('darkgrid')
import matplotlib.pyplot as plt

# Import Deep learning Libraries
import tensorflow as tf
from tensorflow import keras
import tensorflow.image as tfi
from tensorflow.keras.models import Model, load_model
from keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.layers import Conv2D, MaxPool2D, UpSampling2D, concatenate
from tensorflow.keras.layers import Layer, Input, Add, Multiply, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam

# Ignore Warnings
import warnings
warnings.filterwarnings("ignore")

print('modules loaded')

# Function to read and process images
def load_image(image, SIZE):
    return np.round(tfi.resize(img_to_array(load_img(image)) / 255., (SIZE, SIZE)), 4)

def load_images(image_paths, SIZE, mask=False, trim=None):
    if trim is not None:
        image_paths = image_paths[:trim]

    if mask:
        images = np.zeros(shape=(len(image_paths), SIZE, SIZE, 1))
    else:
        images = np.zeros(shape=(len(image_paths), SIZE, SIZE, 3))

    for i, image in enumerate(image_paths):
        img = load_image(image, SIZE)
        if mask:
            images[i] = img[:, :, :1]
        else:
            images[i] = img

    return images

# Functions to display data sample
def show_image(image, title=None, cmap=None, alpha=1):
    plt.imshow(image, cmap=cmap, alpha=alpha)
    if title is not None:
        plt.title(title)
    plt.axis('off')

def show_mask(image, mask, cmap=None, alpha=0.4):
    plt.imshow(image)
    plt.imshow(tf.squeeze(mask), cmap=cmap, alpha=alpha)
    plt.axis('off')

def show_images(imgs, msks):
    plt.figure(figsize=(13,8))

    for i in range(15):
        plt.subplot(3,5,i+1)
        idx = np.random.randint(len(imgs))
        show_mask(imgs[idx], msks[idx], cmap='binary')

    plt.tight_layout()
    plt.show()

# Custom Layers for U-Net
class EncoderBlock(Layer):
    def __init__(self, filters, rate, pooling=True, **kwargs):  # Fixed: __init__ not _init_
        super(EncoderBlock, self).__init__(**kwargs)
        self.filters = filters
        self.rate = rate
        self.pooling = pooling
        self.c1 = Conv2D(filters, kernel_size=3, padding='same', activation='relu', kernel_initializer='he_normal')
        self.drop = Dropout(rate)
        self.c2 = Conv2D(filters, kernel_size=3, padding='same', activation='relu', kernel_initializer='he_normal')
        self.pool = MaxPool2D()

    def call(self, X):
        x = self.c1(X)
        x = self.drop(x)
        x = self.c2(x)
        if self.pooling:
            y = self.pool(x)
            return y, x
        return x

    def get_config(self):
        base_config = super().get_config()
        return {**base_config, "filters": self.filters, "rate": self.rate, "pooling": self.pooling}

class DecoderBlock(Layer):
    def __init__(self, filters, rate, **kwargs):  # Fixed: __init__ not _init_
        super(DecoderBlock, self).__init__(**kwargs)
        self.filters = filters
        self.rate = rate
        self.up = UpSampling2D()
        self.net = EncoderBlock(filters, rate, pooling=False)

    def call(self, X):
        X, skip_X = X
        x = self.up(X)
        c_ = concatenate([x, skip_X])
        x = self.net(c_)
        return x

    def get_config(self):
        base_config = super().get_config()
        return {**base_config, "filters": self.filters, "rate": self.rate}

class AttentionGate(Layer):
    def __init__(self, filters, bn, **kwargs):  # Fixed: __init__ not _init_
        super(AttentionGate, self).__init__(**kwargs)
        self.filters = filters
        self.bn = bn
        self.normal = Conv2D(filters, kernel_size=3, padding='same', activation='relu', kernel_initializer='he_normal')
        self.down = Conv2D(filters, kernel_size=3, strides=2, padding='same', activation='relu', kernel_initializer='he_normal')
        self.learn = Conv2D(1, kernel_size=1, padding='same', activation='sigmoid')
        self.resample = UpSampling2D()
        self.BN = BatchNormalization()

    def call(self, X):
        X, skip_X = X
        x = self.normal(X)
        skip = self.down(skip_X)
        x = Add()([x, skip])
        x = self.learn(x)
        x = self.resample(x)
        f = Multiply()([x, skip_X])
        if self.bn:
            return self.BN(f)
        return f

    def get_config(self):
        base_config = super().get_config()
        return {**base_config, "filters": self.filters, "bn": self.bn}

# Function to plot training history
def plot_training(hist):
    tr_acc = hist.history['accuracy']
    tr_loss = hist.history['loss']
    val_acc = hist.history['val_accuracy']
    val_loss = hist.history['val_loss']
    index_loss = np.argmin(val_loss)
    val_lowest = val_loss[index_loss]
    index_acc = np.argmax(val_acc)
    acc_highest = val_acc[index_acc]
    Epochs = [i+1 for i in range(len(tr_acc))]
    loss_label = f'best epoch= {str(index_loss + 1)}'
    acc_label = f'best epoch= {str(index_acc + 1)}'

    plt.figure(figsize=(20, 8))
    plt.style.use('fivethirtyeight')

    plt.subplot(1, 2, 1)
    plt.plot(Epochs, tr_loss, 'r', label='Training loss')
    plt.plot(Epochs, val_loss, 'g', label='Validation loss')
    plt.scatter(index_loss + 1, val_lowest, s=150, c='blue', label=loss_label)
    plt.title('Training and Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(Epochs, tr_acc, 'r', label='Training Accuracy')
    plt.plot(Epochs, val_acc, 'g', label='Validation Accuracy')
    plt.scatter(index_acc + 1, acc_highest, s=150, c='blue', label=acc_label)
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.tight_layout()
    plt.show()

# --- Section to handle data using metadata.csv ---
SIZE = 256

# ⚠️ DIRECTORY PATH FINDER - Let's help you find the correct path ⚠️
print("=== DIRECTORY PATH FINDER ===")

# Common possible locations for your files
possible_paths = [
    'C:/Users/agraw/Downloads/Flood detector application imp files',
    'C:/Users/agraw/Downloads/flood-detector-application-imp-files',
    'C:/Users/agraw/Downloads/flood_detector_application_imp_files',
    'C:/Users/agraw/Downloads',
    '/content/drive/MyDrive/flood_data',  # Google Colab
    '/kaggle/input/flood-area-segmentation',  # Kaggle
    './flood_data',  # Current directory
    './data',  # Common data folder
    '.',  # Current working directory
]

data_dir = None
found_paths = []

# Check each possible path
for path in possible_paths:
    if os.path.exists(path):
        found_paths.append(path)
        print(f"✓ Found directory: {path}")

        # Check if it has the expected structure
        images_dir = os.path.join(path, 'images')
        masks_dir = os.path.join(path, 'masks')
        csv_file = os.path.join(path, 'metadata.csv')

        has_images = os.path.exists(images_dir)
        has_masks = os.path.exists(masks_dir)
        has_csv = os.path.exists(csv_file)

        print(f"  - Has 'images' folder: {has_images}")
        print(f"  - Has 'masks' folder: {has_masks}")
        print(f"  - Has 'metadata.csv': {has_csv}")

        if has_images and has_masks:
            data_dir = path
            print(f"✓ Using this directory: {path}")
            break
    else:
        print(f"✗ Not found: {path}")

# If no suitable directory found, let's explore what's available
if data_dir is None:
    print("\n=== EXPLORING AVAILABLE DIRECTORIES ===")

    # Check Downloads folder contents
    downloads_path = 'C:/Users/agraw/Downloads'
    if os.path.exists(downloads_path):
        print(f"\nContents of Downloads folder ({downloads_path}):")
        try:
            items = os.listdir(downloads_path)
            for item in sorted(items):
                item_path = os.path.join(downloads_path, item)
                if os.path.isdir(item_path):
                    print(f"  📁 {item}/")
                else:
                    print(f"  📄 {item}")
        except PermissionError:
            print("  Permission denied to list Downloads folder")

    # Check current working directory
    current_dir = os.getcwd()
    print(f"\nCurrent working directory: {current_dir}")
    print("Contents:")
    for item in sorted(os.listdir(current_dir)):
        if os.path.isdir(item):
            print(f"  📁 {item}/")
        else:
            print(f"  📄 {item}")

    # Manual path input
    print("\n" + "="*50)
    print("MANUAL PATH SETUP:")
    print("Please manually set your data directory path below:")
    print("Example formats:")
    print("  data_dir = 'C:/Users/agraw/Downloads/your-actual-folder-name'")
    print("  data_dir = './your-local-folder'")
    print("="*50)

    # Use current directory as fallback
    data_dir = current_dir
    print(f"Using current directory as fallback: {data_dir}")

print(f"\nSelected data directory: {data_dir}")

# Load the CSV file into a pandas DataFrame using the full path
csv_path = os.path.join(data_dir, 'metadata.csv')

try:
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"CSV file not found: {csv_path}")

    df = pd.read_csv(csv_path)
    print(f"Loaded metadata with {len(df)} entries")

    # Create the full file paths by joining the directory with the filenames
    image_paths = [os.path.join(data_dir, 'images', filename) for filename in df['Image']]
    mask_paths = [os.path.join(data_dir, 'masks', filename) for filename in df['Mask']]

    # Verify files exist
    missing_images = [path for path in image_paths if not os.path.exists(path)]
    missing_masks = [path for path in mask_paths if not os.path.exists(path)]

    if missing_images:
        print(f"Warning: {len(missing_images)} image files not found")
        print("First few missing:", missing_images[:3])

    if missing_masks:
        print(f"Warning: {len(missing_masks)} mask files not found")
        print("First few missing:", missing_masks[:3])

except FileNotFoundError as e:
    print(f"Error: {e}")
    print("Trying to find files using glob pattern...")

    # Alternative: use glob to find files directly
    image_dir = os.path.join(data_dir, 'images')
    mask_dir = os.path.join(data_dir, 'masks')

    if not os.path.exists(image_dir):
        print(f"Images directory not found: {image_dir}")
        # Try common image extensions in the main directory
        image_paths = sorted(glob.glob(os.path.join(data_dir, '*.jpg')) +
                           glob.glob(os.path.join(data_dir, '*.jpeg')) +
                           glob.glob(os.path.join(data_dir, '*.png')))
    else:
        image_paths = sorted(glob.glob(os.path.join(image_dir, '*')))

    if not os.path.exists(mask_dir):
        print(f"Masks directory not found: {mask_dir}")
        # Try to find mask files in main directory
        mask_paths = sorted(glob.glob(os.path.join(data_dir, '*mask*')) +
                           glob.glob(os.path.join(data_dir, '*label*')))
    else:
        mask_paths = sorted(glob.glob(os.path.join(mask_dir, '*')))

    print(f"Found {len(image_paths)} images and {len(mask_paths)} masks using glob")

    # Ensure we have matching pairs
    if len(image_paths) != len(mask_paths):
        min_len = min(len(image_paths), len(mask_paths))
        print(f"Warning: Mismatched counts. Using first {min_len} pairs.")
        image_paths = image_paths[:min_len]
        mask_paths = mask_paths[:min_len]

# Verify we have data before proceeding
if len(image_paths) == 0 or len(mask_paths) == 0:
    print("=" * 60)
    print("ERROR: No image or mask files found!")
    print("=" * 60)

    print("DIAGNOSTIC INFORMATION:")
    print(f"Current data_dir: {data_dir}")
    print(f"Looking for images in: {os.path.join(data_dir, 'images')}")
    print(f"Looking for masks in: {os.path.join(data_dir, 'masks')}")
    print(f"Looking for CSV at: {os.path.join(data_dir, 'metadata.csv')}")

    # List what's actually in the directory
    if os.path.exists(data_dir):
        print(f"\nContents of {data_dir}:")
        try:
            items = os.listdir(data_dir)
            if not items:
                print("  (Directory is empty)")
            else:
                for item in sorted(items):
                    item_path = os.path.join(data_dir, item)
                    if os.path.isdir(item_path):
                        # Count files in subdirectories
                        try:
                            sub_items = os.listdir(item_path)
                            print(f"  📁 {item}/ ({len(sub_items)} items)")
                        except:
                            print(f"  📁 {item}/ (can't access)")
                    else:
                        file_size = os.path.getsize(item_path)
                        print(f"  📄 {item} ({file_size} bytes)")
        except Exception as e:
            print(f"  Error listing directory: {e}")
    else:
        print(f"Directory {data_dir} does not exist!")

    print("\n" + "=" * 60)
    print("SOLUTIONS:")
    print("=" * 60)

    print("1. FIND YOUR ACTUAL DATA:")
    print("   - Look for folders containing flood images")
    print("   - Update the data_dir path to point to your actual data")

    print("\n2. CREATE SAMPLE DATA FOR TESTING:")
    print("   - I can create dummy data to test if the model works")

    print("\n3. DOWNLOAD SAMPLE DATASET:")
    print("   - Use a public flood segmentation dataset")

    # Option to create dummy data for testing
    create_dummy = input("\nWould you like me to create dummy data for testing? (y/n): ").lower().strip()

    if create_dummy == 'y':
        print("Creating dummy dataset for testing...")

        # Create directories
        os.makedirs(os.path.join(data_dir, 'images'), exist_ok=True)
        os.makedirs(os.path.join(data_dir, 'masks'), exist_ok=True)

        # Create dummy images and masks
        num_samples = 20
        image_list = []
        mask_list = []

        for i in range(num_samples):
            # Create dummy image (3 channels, RGB)
            dummy_image = np.random.randint(0, 255, (SIZE, SIZE, 3), dtype=np.uint8)
            image_path = os.path.join(data_dir, 'images', f'image_{i:03d}.png')
            cv2.imwrite(image_path, dummy_image)

            # Create dummy mask (binary)
            dummy_mask = np.random.randint(0, 2, (SIZE, SIZE), dtype=np.uint8) * 255
            mask_path = os.path.join(data_dir, 'masks', f'mask_{i:03d}.png')
            cv2.imwrite(mask_path, dummy_mask)

            image_list.append(f'image_{i:03d}.png')
            mask_list.append(f'mask_{i:03d}.png')

        # Create metadata.csv
        dummy_df = pd.DataFrame({
            'Image': image_list,
            'Mask': mask_list
        })
        dummy_df.to_csv(os.path.join(data_dir, 'metadata.csv'), index=False)

        print(f"✓ Created {num_samples} dummy image-mask pairs")
        print(f"✓ Created metadata.csv")

        # Reload the paths
        csv_path = os.path.join(data_dir, 'metadata.csv')
        df = pd.read_csv(csv_path)
        image_paths = [os.path.join(data_dir, 'images', filename) for filename in df['Image']]
        mask_paths = [os.path.join(data_dir, 'masks', filename) for filename in df['Mask']]

        print(f"✓ Ready to train with {len(image_paths)} samples")

    else:
        print("\nPlease:")
        print("1. Find your actual flood detection dataset")
        print("2. Update the data_dir variable to point to the correct location")
        print("3. Ensure your data has this structure:")
        print("   your_data_folder/")
        print("   ├── images/")
        print("   │   ├── image1.jpg")
        print("   │   ├── image2.jpg")
        print("   │   └── ...")
        print("   ├── masks/")
        print("   │   ├── mask1.jpg")
        print("   │   ├── mask2.jpg")
        print("   │   └── ...")
        print("   └── metadata.csv")

        raise ValueError("No training data found. Please check your data directory structure or create dummy data.")

print(f"Found {len(image_paths)} image paths and {len(mask_paths)} mask paths")

# Load images and masks with error handling
print("Loading images and masks...")
try:
    imgs = load_images(image_paths, SIZE)
    msks = load_images(mask_paths, SIZE, mask=True)

    # Ensure masks are binary
    msks = (msks > 0.5).astype(np.float32)

    print(f"Images shape: {imgs.shape}")
    print(f"Masks shape: {msks.shape}")

    # Verify we actually loaded data
    if imgs.shape[0] == 0:
        raise ValueError("No images were successfully loaded!")

    print(f"Successfully loaded {imgs.shape[0]} samples")

    # Show a sample (only if we have data)
    if len(imgs) > 0:
        show_images(imgs, msks)

except Exception as e:
    print(f"Error loading images: {e}")
    print("First few image paths:")
    for i, path in enumerate(image_paths[:5]):
        print(f"  {i+1}. {path} - Exists: {os.path.exists(path)}")
    print("First few mask paths:")
    for i, path in enumerate(mask_paths[:5]):
        print(f"  {i+1}. {path} - Exists: {os.path.exists(path)}")
    raise

# --- Assemble the Full U-Net Model ---
print("Building U-Net model...")
input_ = Input(shape=(SIZE, SIZE, 3))
c1, s1 = EncoderBlock(64, 0.1)(input_)
c2, s2 = EncoderBlock(128, 0.1)(c1)
c3, s3 = EncoderBlock(256, 0.2)(c2)
c4, s4 = EncoderBlock(512, 0.2)(c3)
c5 = EncoderBlock(1024, 0.3, pooling=False)(c4)
d1 = DecoderBlock(512, 0.2)([c5, s4])
d2 = DecoderBlock(256, 0.2)([d1, s3])
d3 = DecoderBlock(128, 0.1)([d2, s2])
d4 = DecoderBlock(64, 0.1)([d3, s1])
output = Conv2D(1, 1, padding='same', activation='sigmoid')(d4)
model = Model(input_, output)

print("Model summary:")
model.summary()

# Compile and Train the Model (only if we have data)
if imgs.shape[0] > 0:
    epochs = 50
    print("Compiling model...")
    model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])

    # Adjust validation split based on data size
    val_split = 0.1 if imgs.shape[0] >= 10 else 0.0
    if val_split == 0.0:
        print("Warning: Using all data for training (no validation split due to small dataset)")

    print(f"Starting training with {imgs.shape[0]} samples...")
    history = model.fit(
        imgs,
        msks,
        batch_size=min(16, imgs.shape[0]),  # Adjust batch size for small datasets
        epochs=epochs,
        validation_split=val_split,
        verbose=1
    )

    # Plot training history
    print("Plotting training history...")
    plot_training(history)

    # Save the model
    print("Saving model...")
    model.save('flood_segmentation_unet.h5')
    print("Model saved successfully!")
else:
    print("ERROR: No data loaded, cannot train model!")

Output hidden; open in https://colab.research.google.com to view.